In [15]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import accuracy_score, classification_report

from sklearn.ensemble import RandomForestClassifier
# from xgboost import XGBClassifier

from sklearn.cluster import KMeans, DBSCAN

In [16]:
df = pd.read_csv("marketing_campaign.csv", sep="\t")

print(df.shape)
print(df.head())

(2240, 29)
     ID  Year_Birth   Education Marital_Status   Income  Kidhome  Teenhome  \
0  5524        1957  Graduation         Single  58138.0        0         0   
1  2174        1954  Graduation         Single  46344.0        1         1   
2  4141        1965  Graduation       Together  71613.0        0         0   
3  6182        1984  Graduation       Together  26646.0        1         0   
4  5324        1981         PhD        Married  58293.0        1         0   

  Dt_Customer  Recency  MntWines  ...  NumWebVisitsMonth  AcceptedCmp3  \
0  04-09-2012       58       635  ...                  7             0   
1  08-03-2014       38        11  ...                  5             0   
2  21-08-2013       26       426  ...                  4             0   
3  10-02-2014       26        11  ...                  6             0   
4  19-01-2014       94       173  ...                  5             0   

   AcceptedCmp4  AcceptedCmp5  AcceptedCmp1  AcceptedCmp2  Complain  \
0   

In [17]:
df.drop("ID", axis=1, inplace=True)

# Fill missing Income values
df["Income"] = df["Income"].fillna(df["Income"].median())

# Convert date column
df["Dt_Customer"] = pd.to_datetime(df["Dt_Customer"], dayfirst=True)

# Create customer age
df["Age"] = 2026 - df["Year_Birth"]

# Drop original columns
df.drop(["Year_Birth"], axis=1, inplace=True)

In [18]:
categorical_cols = ["Education", "Marital_Status"]

le = LabelEncoder()

for col in categorical_cols:
    df[col] = le.fit_transform(df[col])

# Convert date to number of days
df["Customer_Days"] = (
    pd.Timestamp.today() - df["Dt_Customer"]
).dt.days

df.drop("Dt_Customer", axis=1, inplace=True)

In [19]:
X = df.drop("Response", axis=1)
y = df["Response"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [20]:
rf = RandomForestClassifier(
    n_estimators=200,
    random_state=42
)

rf.fit(X_train, y_train)

rf_pred = rf.predict(X_test)

print("\nRandom Forest Accuracy:")
print(accuracy_score(y_test, rf_pred))

print(classification_report(y_test, rf_pred))


Random Forest Accuracy:
0.8861607142857143
              precision    recall  f1-score   support

           0       0.89      0.98      0.94       381
           1       0.79      0.33      0.46        67

    accuracy                           0.89       448
   macro avg       0.84      0.66      0.70       448
weighted avg       0.88      0.89      0.87       448



In [23]:
cluster_features = [
    "Income",
    "MntWines",
    "MntMeatProducts",
    "MntFishProducts",
    "MntGoldProds",
    "NumWebPurchases",
    "NumStorePurchases",
    "Age"
]

X_cluster = df[cluster_features]

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_cluster)

kmeans = KMeans(
    n_clusters=4,
    random_state=42
)

df["KMeans_Cluster"] = kmeans.fit_predict(X_scaled)

print("\nKMeans Cluster Counts")
print(df["KMeans_Cluster"].value_counts())


KMeans Cluster Counts
KMeans_Cluster
0    1062
1     643
3     274
2     261
Name: count, dtype: int64


c:\Users\Sreeja Reddy\anaconda3\Lib\site-packages\sklearn\cluster\_kmeans.py:1419: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=9.
  warnings.warn(


In [24]:
dbscan = DBSCAN(
    eps=1.2,
    min_samples=8
)

df["DBSCAN_Cluster"] = dbscan.fit_predict(X_scaled)

print("\nDBSCAN Cluster Counts")
print(df["DBSCAN_Cluster"].value_counts())


DBSCAN Cluster Counts
DBSCAN_Cluster
 0    1556
-1     674
 1      10
Name: count, dtype: int64


In [25]:
df.to_csv(
    "customer_intelligence_output.csv",
    index=False
)

print("\nProject Completed Successfully!")


Project Completed Successfully!
